# DCC 2026 Mission 2 — 신고자 / 119대원 분류

Google Colab에서 Mission 2의 MFCC + RandomForest 기준선을 학습하고 검증합니다. 각 발화는 JSON의 `startAt`, `endAt` 경계로만 자르며, 텍스트와 발화 순서는 사용하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Drive 안의 실제 폴더명에 맞춰 한 번만 수정하세요.
DATA_ROOT = Path('/content/drive/MyDrive/data(inho)')
CODE_DIR = Path('/content/drive/MyDrive/mission2')
MODEL_DIR = Path('/content/drive/MyDrive/saved_models/mission2')
MODEL_PATH = MODEL_DIR / 'mission2_mfcc_rf.joblib'
OUTPUT_DIR = Path('/content/drive/MyDrive/outputs')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('데이터:', DATA_ROOT.exists(), DATA_ROOT)
print('코드:', CODE_DIR.exists(), CODE_DIR)
print('모델 저장:', MODEL_DIR)

## 준비

Drive의 `mission2` 폴더에 저장소의 `src/mission2/` 안 파일 3개(`common.py`, `train_mfcc_rf.py`, `inference.py`)를 업로드한 뒤 아래 셀을 실행합니다.

In [ ]:
!pip -q install librosa scikit-learn joblib tqdm soundfile

required_files = ['common.py', 'train_mfcc_rf.py', 'inference.py']
missing = [name for name in required_files if not (CODE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'CODE_DIR에 업로드되지 않은 파일: {missing}')
print('Mission 2 코드 준비 완료')

## 데이터 무결성 확인

서울 Training/Validation의 WAV와 JSON 개수 및 파일명 매칭을 확인합니다. Validation은 이후 평가에만 사용합니다.

In [ ]:
def collect_paths(split: str):
    audio_name, label_name = (('TS_서울_구급', 'TL_서울_구급') if split == 'Training'
                              else ('VS_서울_구급', 'VL_서울_구급'))
    wavs, labels = [], []
    for part in sorted(path for path in DATA_ROOT.iterdir() if path.is_dir()):
        base = part / '대학부 데이터' / split
        wavs.extend((base / '1.원천데이터' / audio_name).rglob('*.wav'))
        labels.extend((base / '2.라벨링데이터' / label_name).rglob('*.json'))
    return wavs, labels

for split in ('Training', 'Validation'):
    wavs, labels = collect_paths(split)
    wav_stems, label_stems = {p.stem for p in wavs}, {p.stem for p in labels}
    print(f'{split}: WAV={len(wavs):,}, JSON={len(labels):,}, WAV만={len(wav_stems-label_stems):,}, JSON만={len(label_stems-wav_stems):,}')

In [ ]:
# Training 정답 speaker의 클래스 수만 확인합니다. text나 발화 순서는 사용하지 않습니다.
import json
from collections import Counter

_, train_labels = collect_paths('Training')
speaker_counts = Counter()
utterance_count = 0
for path in train_labels:
    with path.open(encoding='utf-8') as file:
        annotation = json.load(file)
    utterances = annotation.get('utterances', annotation.get('utterences', []))
    for utterance in utterances:
        speaker = str(utterance.get('speaker', utterance.get('Speaker', ''))).strip()
        if speaker in {'0', '1'}:
            speaker_counts[speaker] += 1
            utterance_count += 1

print('유효 발화 수:', f'{utterance_count:,}')
print('119대원(0):', f"{speaker_counts['0']:,}")
print('신고자(1):', f"{speaker_counts['1']:,}")

## 학습 및 Validation 평가

특징 추출은 시간이 걸립니다. 현재 기준선은 MFCC와 음량·스펙트럼 통계값을 사용하며, 학습은 Training 데이터만 이용합니다.

In [ ]:
%cd $CODE_DIR
!python train_mfcc_rf.py \
  --data_root "$DATA_ROOT" \
  --ckpt_path "$MODEL_PATH" \
  --n_estimators 500 \
  --seed 42

## Validation CSV 생성 및 형식 확인

아래는 Validation 데이터에 대한 제출 형식 CSV를 만드는 확인 단계입니다. 성능 수치는 학습 명령에서 이미 출력됩니다.

In [ ]:
# 첫 번째 Validation part를 사용해 제출 CSV 형식을 점검합니다.
first_part = next(path for path in sorted(DATA_ROOT.iterdir()) if path.is_dir())
val_base = first_part / '대학부 데이터' / 'Validation'
VAL_WAV_DIR = val_base / '1.원천데이터' / 'VS_서울_구급'
VAL_LABEL_DIR = val_base / '2.라벨링데이터' / 'VL_서울_구급'
VAL_OUTPUT = OUTPUT_DIR / 'mission2_validation_sample.csv'

%cd $CODE_DIR
!python inference.py \
  --audio_dir "$VAL_WAV_DIR" \
  --label_dir "$VAL_LABEL_DIR" \
  --ckpt_path "$MODEL_PATH" \
  --output "$VAL_OUTPUT"

import pandas as pd
result = pd.read_csv(VAL_OUTPUT)
display(result.head())
print('행 수:', len(result))
assert list(result.columns) == ['audio file name', 'startAt', 'endAt', 'speaker']
assert set(result['speaker'].unique()).issubset({0, 1})

## 최종 제출

주최 측이 제공한 평가 WAV/JSON 폴더로 아래 경로만 교체해 한 번 실행합니다. 평가 데이터에서는 `speaker` 정답을 읽지 않으며, `startAt`, `endAt`으로 구간만 자릅니다.

```bash
python inference.py --audio_dir {wav_folder} --label_dir {json_folder} --ckpt_path {checkpoint_file} --output ./outputs/mission2.csv
```